# # Kapitel 3 - Koduppgift 16

In [38]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

#läs in data från csv filen, det finns en kolumn som heter Unnamed och tittar man i excel så finns det kolumner som saknar data. 
# Antingen ersätter man den tomma kolumner med median värde eller så tar man bort raderna. Tar man bort rader så förlorar man värdefull data som kan vara användbar.
# Raderar man rader med tom data så kommer testet troligtvist att köras snabbare. 
df = pd.read_csv("diamonds.csv")
print(df.head())
print()

   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2           3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07   
3           4   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23   
4           5   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35   

      z  
0  2.43  
1  2.31  
2  2.31  
3  2.63  
4  2.75  



In [39]:
# Ta bort onödig kolumn, "Unnamed: 0" eftersom den bara innehåller index
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Ta bort rader där dimensioner är 0 eftersom dessa värden inte är realistiska
df = df[df["x"] != 0]
df = df[df["y"] != 0]
df = df[df["z"] != 0]

print(df.head())

   carat      cut color clarity  depth  table  price     x     y     z
0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75


In [40]:
# variabler till numeriska med dummy encoding
df = pd.get_dummies(df, columns=["cut", "color", "clarity"], drop_first=True)

In [41]:
# Dela upp data i x och y
X = df.drop(columns=["price"])
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print()

X shape: (53920, 23)
y shape: (53920,)



In [42]:
# Dela upp i träningsdata och testdata
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [43]:
# Skapa modeller
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42)

# Kör korsvalidering för att jämföra modeller
cv_lr = cross_validate(lr, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)
cv_rf = cross_validate(rf, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)

# Gör om till positiva RMSE-värden
lr_score = -np.mean(cv_lr['test_score'])
rf_score = -np.mean(cv_rf['test_score'])

print(f"Linear Regression: {lr_score:.2f}")
print(f"Random Forest: {rf_score:.2f}")
print()

Linear Regression: 1191.17
Random Forest: 643.67



In [44]:
# Välj bästa modell baserat på resultat
if np.mean(cv_lr['test_score']) > np.mean(cv_rf['test_score']):
    print("Bästa modell: Linear Regression")
    best_model = LinearRegression()

elif np.mean(cv_lr['test_score']) == np.mean(cv_rf['test_score']):
    print("Modellerna presterar lika bra, väljer Linear Regression")
    best_model = LinearRegression()

else:
    print("Bästa modell: Random Forest")
    best_model = RandomForestRegressor(random_state=42)

Bästa modell: Random Forest


In [45]:
# Träna den bästa modellen på hela träningsdatan
best_model.fit(X_train, y_train)

# Gör prediktioner
y_pred = best_model.predict(X_test)

# Utvärdera modellen med RMSE
rmse = root_mean_squared_error(y_test, y_pred)
print(f"RMSE: {rmse:.2f}")
print()

RMSE: 589.42



##### I denna uppgift genomförs ett komplett maskininlärningsflöde för att modellera diamantpriser. Data läses in från en CSV-fil och rensas genom att ta bort en onödig kolumn samt orimliga värden. Kategoriska variabler omvandlas till numeriska med hjälp av dummy encoding.

##### Därefter delas datan upp i träningsdata och testdata. Två modeller, Linear Regression och Random Forest, tränas och jämförs med hjälp av korsvalidering och RMSE som mått. Den modell som presterar bäst väljs, tränas på hela träningsdatan och utvärderas slutligen på testdatan.